In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df_full = pd.read_csv("speed_dirs_test.csv")

ids = df_full["id"].unique()
df_individuals = []

for i in ids:
    df_individuals.append(df_full[df_full.id == i])

In [3]:
df_individuals[8]

,Unnamed: 0,image_filename,width,height,class,x1,y1,x2,y2,x_center,y_center,id,distance,speed,direction
14392,8,wt_25c_3_dpf_fert_04_24_20260001_png.rf.kqofjt...,11,13,Planula,353,352,364,365,358.5,358.5,8,NaN,NaN,NaN
14393,31,wt_25c_3_dpf_fert_04_24_20260002_png.rf.O1pPHe...,13,14,Planula,352,352,365,366,358.5,359.0,8,8.964143,268.924303,4.712389
14394,44,wt_25c_3_dpf_fert_04_24_20260003_png.rf.U5UqKw...,13,13,Planula,353,354,366,367,359.5,360.5,8,32.410913,972.327384,0.978631
14395,74,wt_25c_3_dpf_fert_04_24_20260004_png.rf.XWE4Zm...,14,13,Planula,353,355,367,368,360.0,361.5,8,20.080826,602.424789,1.103537
14396,96,wt_25c_3_dpf_fert_04_24_20260005_png.rf.FkIViI...,15,13,Planula,354,356,369,369,361.5,362.5,8,32.523353,975.700600,0.583854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16186,35893,wt_25c_3_dpf_fert_04_24_20261795_png.rf.vdHfo7...,13,13,Planula,253,208,266,221,259.5,214.5,8,32.410913,972.327384,5.304555
16187,35913,wt_25c_3_dpf_fert_04_24_20261796_png.rf.mhPpND...,13,13,Planula,254,208,267,221,260.5,214.5,8,18.090452,542.713568,0.000000
16188,35932,wt_25c_3_dpf_fert_04_24_20261797_png.rf.phpJeT...,13,13,Planula,255,207,268,220,261.5,213.5,8,25.469353,764.080584,5.502289
16189,35951,wt_25c_3_dpf_fert_04_24_20261798_png.rf.6ucxAG...,12,13,Planula,256,207,268,220,262.0,213.5,8,9.045226,271.356784,0.000000


Note that the above individual rapidly changes direction from 4.7 (south) to 0.9 (northwest) and stays there for a few frames. In other words, we can look for these rapid changes in direction to find if things are taking loops. One way to do this is with entropy: essentially, by "binning" the directions we take and calculating the negative log sum over their distribution, we can find out how much each direction is taken. The most disordered distribution would have an equal amount of all directions. For example, with 8 directions in equal probability, this is $-8 * \frac{1}{8} \log (\frac{1}{8})$, or $- \log (\frac{1}{8})$, which comes out to 3. Meanwhile, if we had 8 instances of just one direction, this is an entropy of 0. 

It would stand that animals moving in a straight line have lower local entropy within some number of frames, while animals looping would have much higher local entropy. So we can calculate entropy totals across frame windows to find a "loop score" that allows us to test how much something is looping. 

Here's the plan I have so far, then:
1. Determine how much things move in total. If an animal does not move much at all (for example, less than 10000 microns total, which is often simply due to shifts in detection across ~2000 frames) we don't calculate a loop score.
2. Given a window size $w$, split up the moving animals' dataframes into windows of maximum size $w$. Within each window, calculate the direction entropy.
3. Add up the direction entropies across the windows. Since all organisms are treated the same, we don't need to do averages, but you might need to do averages if you did a different comparison elsewhere or had a video with more or fewer frames. 
4. Any organism with an entropy total above a certain threshold can be determined to be a spiraler.

This requires som helpers:
1. Given a bunch of directions between 0 and $2\pi$ and a number of bins $n$, find the counts for each direction falling into the $n$ evenly spaced bins. This is just done with `np.histogram`.
2. Given the bins from above, determine the entropy. Straightforward enough. 

In [4]:
boundaries = np.linspace(0, 2*np.pi, num=17)
boundaries = boundaries - 0.5 * boundaries[1]
boundaries[0] = 0
boundaries = np.append(boundaries, 2*np.pi)
print(boundaries)

hist, bins = np.histogram(df_individuals[8]["direction"], bins = boundaries)
print(hist)
hist[0] = hist[0] + hist[-1]
hist = hist[:-1]
print(hist)

[0.         0.19634954 0.58904862 0.9817477  1.37444679 1.76714587
 2.15984495 2.55254403 2.94524311 3.33794219 3.73064128 4.12334036
 4.51603944 4.90873852 5.3014376  5.69413668 6.08683577 6.28318531]
[183  56 100  52 214  46  87  79 194  90  99  74 171  72 107  69   0]
[183  56 100  52 214  46  87  79 194  90  99  74 171  72 107  69]


In [5]:
def bin_dirs(df, num_bins=9):
    boundaries = np.linspace(0, 2*np.pi, num=num_bins)
    boundaries = boundaries - 0.5 * boundaries[1]
    boundaries[0] = 0
    boundaries = np.append(boundaries, 2*np.pi)

    hist, bins = np.histogram(df["direction"], bins = boundaries)
    hist[0] = hist[0] + hist[-1]
    hist = hist[:-1]

    return hist

def entropy(bins):
    total = np.sum(bins)
    probs = bins / total

    # log_probs = np.log2(probs)
    # p_log_probs = probs * log_probs

    # return -1 * np.sum(p_log_probs)
    ent = 0
    for i in range(len(probs)):
        if bins[i] == 0:
            continue
        logp = np.log2(probs[i])
        plogp = probs[i] * logp
        ent -= plogp

    return ent

In [6]:
hist = bin_dirs(df_individuals[8])
print(hist)
print(entropy(hist))

[212 181 244 169 242 219 215 211]
2.9901645815940765


In [7]:
#now we calculate entropy across windows
def find_window_entropy(df, window_size = 30, num_dirs = 8):
    indices = np.linspace(0, len(df) + 1, window_size)

    total_entropy = 0
    for i in range(len(indices) - 1):
        current_df = df.iloc[int(indices[i]):int(indices[i+1])]
        local_entropy = entropy(bin_dirs(current_df, num_bins=num_dirs+1))
        total_entropy += local_entropy

    return total_entropy

In [8]:
print(find_window_entropy(df_individuals[18]))

51.84150746973305


/var/folders/px/b7vc3nh913zb_m0x36ncftj00000gn/T/ipykernel_21739/1103273023.py:15: RuntimeWarning: invalid value encountered in divide
  probs = bins / total


In [9]:
for i in range(len(df_individuals)):
    print("Individual " + str(i) + ": " + str(find_window_entropy(df_individuals[i], window_size=60)))


Individual 0: 87.75115482876524
Individual 1: 55.836755363317444
Individual 2: 89.72736512885477
Individual 3: 98.07626861026806
Individual 4: 72.14903225730647
Individual 5: 68.65398736105061
Individual 6: 77.62008642170441
Individual 7: 108.52900904437804
Individual 8: 140.06877556380672
Individual 9: 32.859129350426755
Individual 10: 111.2716688883884
Individual 11: 85.13959822790397
Individual 12: 138.8910314301631
Individual 13: 86.07991045997022
Individual 14: 85.97310719931244
Individual 15: 170.12493184399665
Individual 16: 104.9117641234977
Individual 17: 107.4995767291645
Individual 18: 84.17895014928749
Individual 19: 132.3659571543161


/var/folders/px/b7vc3nh913zb_m0x36ncftj00000gn/T/ipykernel_21739/1103273023.py:15: RuntimeWarning: invalid value encountered in divide
  probs = bins / total
